In [ ]:
# DS605 - Lab Assignment 1
## Web Scraping and Data Analysis using Scrapy
##**Name:** Ummehani Khatri
##**Student ID:** 202618029
##**Course:** DS605

SyntaxError: invalid syntax (2034583926.py, line 3)

In [ ]:
import scrapy
import pandas as pd
import matplotlib.pyplot as plt
from wordcloud import WordCloud

print("Libraries imported successfully")

In [ ]:
## Task 1 - Web Scraping

Scraped data from the first five pages of https://books.toscrape.com using Scrapy.

Extracted:
- Title
- Category
- Price
- Rating
- Availability
- Description
- UPC
- Number of Reviews
- Product URL

In [ ]:
import scrapy


class BooksSpider(scrapy.Spider):
    name = "books"
    allowed_domains = ["books.toscrape.com"]
    start_urls = ["https://books.toscrape.com/catalogue/page-1.html"]

    def parse(self, response):

        # Visit every book on the current page
        for book in response.css("article.product_pod"):
            detail_url = response.urljoin(book.css("h3 a::attr(href)").get())
            yield scrapy.Request(detail_url, callback=self.parse_book)

        # Go only to pages 2,3,4,5
        current_page = int(
            response.url.split("page-")[-1].replace(".html", "")
        )

        if current_page < 5:
            next_page = response.css("li.next a::attr(href)").get()
            if next_page:
                yield response.follow(next_page, callback=self.parse)

    def parse_book(self, response):

        availability = "".join(
            response.css("p.instock.availability::text").getall()
        ).strip()

        category = response.css("ul.breadcrumb li a::text").getall()
        if len(category) >= 3:
            category = category[2]
        else:
            category = ""

        description = response.css("#product_description + p::text").get()

        yield {
            "title": response.css("div.product_main h1::text").get(),
            "category": category,
            "price": response.css("p.price_color::text").get(),
            "rating": response.css("p.star-rating::attr(class)").get().replace("star-rating ", ""),
            "availability": availability,
            "description": description,
            "upc": response.xpath('//th[text()="UPC"]/following-sibling::td/text()').get(),
            "number_of_reviews": response.xpath('//th[text()="Number of reviews"]/following-sibling::td/text()').get(),
            "product_url": response.url,
        }

In [ ]:
## Task 2 - Data Preprocessing

Performed preprocessing using Pandas:

- Removed duplicates
- Filled missing values
- Converted price to numeric
- Converted ratings to numeric
- Created new features

In [ ]:
import pandas as pd

# Load data
df = pd.read_csv("books.csv")

# Remove extra spaces
df = df.apply(lambda col: col.str.strip() if col.dtype == "object" else col)

# Remove duplicate books using UPC
df = df.drop_duplicates(subset="upc")

# Fill missing descriptions
df["description"] = df["description"].fillna("No Description")

# Convert price (£51.77 -> 51.77)
df["price"] = df["price"].str.replace("£", "", regex=False).astype(float)

# Convert ratings to numbers
rating_map = {
    "One": 1,
    "Two": 2,
    "Three": 3,
    "Four": 4,
    "Five": 5
}
df["rating"] = df["rating"].map(rating_map)

# Extract stock count
df["stock"] = df["availability"].str.extract(r'(\d+)').fillna(0).astype(int)

# New Features
df["description_word_count"] = df["description"].apply(lambda x: len(str(x).split()))

df["price_band"] = pd.cut(
    df["price"],
    bins=[0, 20, 40, 60, 100],
    labels=["Cheap", "Medium", "Expensive", "Premium"]
)

df["value_score"] = df["rating"] / df["price"]

# Save cleaned data
df.to_csv("books_cleaned.csv", index=False)

print(df.head())

print("\nTotal Records:", len(df))
print("Missing Values:\n", df.isnull().sum())
print("Duplicate UPC:", df["upc"].duplicated().sum())

In [ ]:
## Task 3 - Data Visualization

Generated:

- Price Distribution
- Rating Distribution
- Average Price by Category
- Price vs Rating
- Word Cloud

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from wordcloud import WordCloud

# Load cleaned data
df = pd.read_csv("books_cleaned.csv")

# Graph 1: Price Distribution
plt.figure(figsize=(8,5))
plt.hist(df["price"], bins=10)
plt.title("Price Distribution")
plt.xlabel("Price")
plt.ylabel("Number of Books")
plt.savefig("price_distribution.png")
plt.close()

# Graph 2: Rating Distribution
plt.figure(figsize=(8,5))
df["rating"].value_counts().sort_index().plot(kind="bar")
plt.title("Rating Distribution")
plt.xlabel("Rating")
plt.ylabel("Count")
plt.savefig("rating_distribution.png")
plt.close()

# Graph 3: Average Price by Category
plt.figure(figsize=(10,6))
df.groupby("category")["price"].mean().sort_values().plot(kind="bar")
plt.title("Average Price by Category")
plt.ylabel("Average Price")
plt.tight_layout()
plt.savefig("average_price_category.png")
plt.close()

# Graph 4: Price vs Rating
plt.figure(figsize=(8,5))
plt.scatter(df["price"], df["rating"])
plt.title("Price vs Rating")
plt.xlabel("Price")
plt.ylabel("Rating")
plt.savefig("price_vs_rating.png")
plt.close()

# Word Cloud
text = " ".join(df["description"].astype(str))

wc = WordCloud(width=1200, height=600, background_color="white").generate(text)

plt.figure(figsize=(12,6))
plt.imshow(wc, interpolation="bilinear")
plt.axis("off")
plt.savefig("wordcloud.png")
plt.close()

print("All graphs created successfully!")

In [ ]:
# Conclusion

Successfully scraped 100 books from the first five pages of the website, preprocessed the dataset, created new features, and generated visualizations for analysis.